# Diffusion Distance Based Clustering — MSigDB

# Import

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import os
import pickle
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components


from ddbc_functions import (
    DDBCConfig,
    big_objects,
    row_stochastic_check,
    build_kNN,
    build_layer_adjacency_matrices,
    build_multilayer_transition_matrix,
    build_row_aggregation_matrix,
    build_distinct_gene_mapping,
    compute_and_save_diffusion_distance,
    compute_and_save_average_transition_matrix,
    compute_stationary_layer_weights,
    load_bootstrap_scores,
    load_diffusion_distance,
    load_matrices,
    plot_stationary_layer_weights,
    run_bootstrap_robustness,
    run_clustering,
    select_communities,
    stationary_distribution,
)

# Configuration

In [ ]:
config = DDBCConfig(
    disease="NONE",
    average_t=[2, 4, 6, 8],
    num_neighbors=400,
    resolution=1.3,
    size_cap=100,
    score_cap=0,
    n_boots=1000,
    check_every=50,
    tol=0.01,
    patience=2,
    sampling_pct=0.8,
    leiden_seed=42,
)

# Load MSigDB

In [ ]:
os.makedirs(config.output_directory, exist_ok=True)

matrices = load_matrices(config)
MSIGDB_adjacency_matrix = (
    build_layer_adjacency_matrices(matrices)
)

# Build the MSigDB Transition Matrix

In [ ]:
# Save MSIGDB Adjacency Matrix
sp.save_npz("output/msigdb_adjacency_matrix.npz",MSIGDB_adjacency_matrix,compressed=False)

## Construct and validate P

In [ ]:
P = MSIGDB_adjacency_matrix
del MSIGDB_adjacency_matrix
gc.collect()

num_genes = P.shape[0]
assert row_stochastic_check(P)

support = sp.csr_matrix(P > 0)
n_components, component_labels = connected_components(
    support,
    directed=False,
)
assert n_components == 1

## Stationary distribution

In [ ]:
pi = stationary_distribution(
    P,
    tol=config.stationary_tol,
    maxit=config.stationary_maxit,
    seed=config.stationary_seed,
)

# Matrix-Free Method

In [ ]:
P_t_avg = compute_and_save_average_transition_matrix(
    P,
    config,
    None,
    None,
)

In [ ]:
D_avg = compute_and_save_diffusion_distance(P_t_avg, config)

In [ ]:
del P_t_avg
gc.collect()

# Construct kNN

In [ ]:
kNN_adjacency_matrix, kNN_graph = build_kNN(
    D_avg,
    k=config.num_neighbors,
    sym_method="average",
)
kNN_adjacency_matrix.data.mean()

In [ ]:
with open(f"{config.output_directory}/result_graph.pkl", "wb") as file:
    pickle.dump(kNN_graph, file)

# Leiden Clustering

In [ ]:
labels, score, communities = run_clustering(
    kNN_adjacency_matrix,
    config,
)

# Select Communities

In [ ]:
communities_selected = select_communities(kNN_graph, communities, config)
top_m = len(communities_selected)
print(top_m)

# Robustness Analysis

In [ ]:
os.makedirs(config.graph_directory, exist_ok=True)

In [ ]:
big_objects()
ari_scores = run_bootstrap_robustness(kNN_adjacency_matrix, config)

In [ ]:
if "ari_scores" not in locals():
    ari_scores = load_bootstrap_scores(config)

median_ari = np.median(ari_scores)
mean_ari = np.mean(ari_scores)
std_ari = np.std(ari_scores)
print("Median ARI:", median_ari)
print("Mean ARI:", mean_ari)
print("STD of ARI:", std_ari)

In [ ]:
font_size = 20
tick_font_size = 16
plt.rcParams.update(
    {
        "font.size": font_size,
        "axes.titlesize": font_size,
        "axes.labelsize": font_size,
        "xtick.labelsize": tick_font_size,
        "ytick.labelsize": tick_font_size,
        "legend.fontsize": font_size,
        "figure.titlesize": font_size,
        "legend.loc": "best",
    }
)

plt.figure(figsize=(6, 4))
plt.hist(ari_scores, bins=15, edgecolor="black")
plt.axvline(
    median_ari,
    linestyle="--",
    linewidth=2,
    label=f"Median = {median_ari:.3f}",
)
plt.xlabel("Adjusted Rand Index (ARI)")
plt.ylabel("Sample Count")
plt.legend(fontsize=14)
plt.tight_layout()
plt.savefig(
    f"{config.graph_directory}/ari_stability_plot.png",
    dpi=300,
)
plt.show()